In [ ]:
# =====================================================
# PHASE 1 - PART 1
# ENTERPRISE DATA GENERATION
# =====================================================

!pip install -q faker

import pandas as pd
import numpy as np
import sqlite3

from faker import Faker

fake = Faker()

np.random.seed(42)

# =====================================================
# CONFIG
# =====================================================

NUM_EMPLOYEES = 5000
NUM_PROJECTS = 1000

# =====================================================
# MASTER DATA
# =====================================================

skills = [
    "Power BI",
    "SQL",
    "Python",
    "Excel",
    "Azure",
    "Fabric",
    "Data Analytics",
    "Business Analysis",
    "Tableau",
    "Alteryx",
    "Snowflake",
    "Databricks"
]

grades = [
    "Analyst",
    "Senior Analyst",
    "Associate",
    "Senior Associate",
    "Manager"
]

locations = [
    "Pune",
    "Bangalore",
    "Hyderabad",
    "Mumbai",
    "Chennai",
    "Kolkata",
    "Gurugram"
]

# =====================================================
# EMPLOYEE PERSONAS
# =====================================================

personas = np.random.choice(

    [
        "High Performer",
        "Average",
        "Bench",
        "Future Leader",
        "Burnout Risk"
    ],

    NUM_EMPLOYEES,

    p=[
        0.20,
        0.45,
        0.15,
        0.10,
        0.10
    ]

)

# =====================================================
# EMPLOYEES
# =====================================================

employee_rows = []

for i in range(NUM_EMPLOYEES):

    persona = personas[i]

    grade = np.random.choice(
        grades,
        p=[
            0.35,
            0.30,
            0.15,
            0.15,
            0.05
        ]
    )

    experience = np.random.randint(
        1,
        15
    )

    age = np.random.randint(
        22,
        50
    )

    skill = np.random.choice(
        skills
    )

    location = np.random.choice(
        locations
    )

    # ---------------------------------

    if persona == "High Performer":

        performance = round(
            np.random.uniform(
                4.2,
                5.0
            ),
            1
        )

        training = np.random.randint(
            50,
            120
        )

        salary = np.random.randint(
            1200000,
            2800000
        )

    elif persona == "Future Leader":

        performance = round(
            np.random.uniform(
                4.0,
                4.8
            ),
            1
        )

        training = np.random.randint(
            60,
            140
        )

        salary = np.random.randint(
            1500000,
            3500000
        )

    elif persona == "Bench":

        performance = round(
            np.random.uniform(
                2.0,
                3.5
            ),
            1
        )

        training = np.random.randint(
            0,
            25
        )

        salary = np.random.randint(
            400000,
            900000
        )

    elif persona == "Burnout Risk":

        performance = round(
            np.random.uniform(
                4.0,
                5.0
            ),
            1
        )

        training = np.random.randint(
            0,
            15
        )

        salary = np.random.randint(
            1000000,
            2500000
        )

    else:

        performance = round(
            np.random.uniform(
                3.0,
                4.2
            ),
            1
        )

        training = np.random.randint(
            20,
            60
        )

        salary = np.random.randint(
            600000,
            1800000
        )

    employee_rows.append([

        f"E{i+1:05}",

        fake.name(),

        grade,

        skill,

        location,

        experience,

        age,

        salary,

        performance,

        training,

        persona

    ])

employees = pd.DataFrame(

    employee_rows,

    columns=[

        "EmployeeID",
        "EmployeeName",
        "Grade",
        "Skill",
        "Location",
        "ExperienceYears",
        "Age",
        "Salary",
        "PerformanceRating",
        "TrainingHours",
        "Persona"

    ]

)

print("Employees Created")

# =====================================================
# PROJECTS
# =====================================================

clients = [
    f"Client_{i}"
    for i in range(
        1,
        301
    )
]

project_rows = []

for i in range(NUM_PROJECTS):

    revenue = np.random.randint(
        5000000,
        150000000
    )

    margin_pct = np.random.uniform(
        0.15,
        0.45
    )

    cost = int(
        revenue *
        (1 - margin_pct)
    )

    project_rows.append([

        f"P{i+1:05}",

        np.random.choice(
            clients
        ),

        revenue,

        cost,

        round(
            margin_pct*100,
            2
        ),

        np.random.choice(
            [
                "High",
                "Medium",
                "Low"
            ],
            p=[
                0.25,
                0.50,
                0.25
            ]
        )

    ])

projects = pd.DataFrame(

    project_rows,

    columns=[

        "ProjectID",
        "ClientName",
        "Revenue",
        "Cost",
        "MarginPct",
        "Priority"

    ]

)

print("Projects Created")

from datetime import datetime

# =====================================================
# ALLOCATIONS
# =====================================================

allocation_rows = []

for idx,row in employees.iterrows():

    emp = row["EmployeeID"]
    persona = row["Persona"]

    if persona == "Bench":
        project_count = np.random.randint(0,2)

    elif persona == "Burnout Risk":
        project_count = np.random.randint(3,5)

    elif persona == "High Performer":
        project_count = np.random.randint(2,4)

    elif persona == "Future Leader":
        project_count = np.random.randint(2,5)

    else:
        project_count = np.random.randint(1,3)

    if project_count == 0:

        allocation_rows.append([
            emp,
            None,
            0,
            0,
            "Non-Billable"
        ])

    else:

        for _ in range(project_count):

            if persona == "Burnout Risk":
                alloc_pct = np.random.choice(
                    [75,100],
                    p=[0.3,0.7]
                )

            elif persona == "Bench":
                alloc_pct = np.random.choice(
                    [0,25,50],
                    p=[0.4,0.4,0.2]
                )

            else:
                alloc_pct = np.random.choice(
                    [50,75,100]
                )

            allocation_rows.append([

                emp,

                np.random.choice(
                    projects["ProjectID"]
                ),

                alloc_pct,

                int(alloc_pct*0.4),

                "Billable"

            ])

allocations = pd.DataFrame(

    allocation_rows,

    columns=[
        "EmployeeID",
        "ProjectID",
        "AllocationPct",
        "WeeklyHours",
        "BillableFlag"
    ]

)

print("Allocations Created")

# =====================================================
# TIMESHEET
# =====================================================

months = pd.date_range(
    "2025-01-01",
    periods=12,
    freq="MS"
)

timesheet_rows = []

for idx,row in employees.iterrows():

    emp = row["EmployeeID"]
    persona = row["Persona"]

    for month in months:

        available = 160

        if persona == "High Performer":

            billable = np.random.randint(
                135,
                160
            )

        elif persona == "Future Leader":

            billable = np.random.randint(
                120,
                150
            )

        elif persona == "Bench":

            billable = np.random.randint(
                20,
                90
            )

        elif persona == "Burnout Risk":

            billable = np.random.randint(
                145,
                160
            )

        else:

            billable = np.random.randint(
                80,
                130
            )

        leave = np.random.randint(
            0,
            16
        )

        training = np.random.randint(
            0,
            12
        )

        timesheet_rows.append([

            emp,
            month,
            available,
            billable,
            leave,
            training

        ])

timesheet = pd.DataFrame(

    timesheet_rows,

    columns=[
        "EmployeeID",
        "Month",
        "AvailableHours",
        "BillableHours",
        "LeaveHours",
        "TrainingHours"
    ]

)

print("Timesheet Created")

# =====================================================
# UTILIZATION
# =====================================================

util = timesheet.groupby(
    "EmployeeID"
).agg({
    "BillableHours":"sum",
    "AvailableHours":"sum"
}).reset_index()

util["UtilizationPct"] = (

    util["BillableHours"]

    /

    util["AvailableHours"]

) * 100

employees = employees.merge(
    util[
        [
            "EmployeeID",
            "UtilizationPct"
        ]
    ],
    on="EmployeeID"
)

# =====================================================
# ATTRITION
# =====================================================

employees["AttritionRisk"] = np.where(

    (
        employees["Persona"] == "Bench"
    )

    |

    (
        employees["PerformanceRating"] < 3
    ),

    1,

    0

)

# =====================================================
# BURNOUT
# =====================================================

employees["BurnoutRisk"] = np.where(

    (
        employees["Persona"] == "Burnout Risk"
    )

    |

    (
        employees["UtilizationPct"] > 90
    ),

    1,

    0

)

# =====================================================
# RESOURCE HEALTH SCORE
# =====================================================

employees["ResourceHealthScore"] = (

    employees["PerformanceRating"] * 20

    +

    employees["UtilizationPct"] * 0.5

    +

    employees["TrainingHours"] * 0.2

)

# =====================================================
# SQL DATABASE
# =====================================================

conn = sqlite3.connect(
    "workforce.db"
)

employees.to_sql(
    "Employees",
    conn,
    if_exists="replace",
    index=False
)

projects.to_sql(
    "Projects",
    conn,
    if_exists="replace",
    index=False
)

allocations.to_sql(
    "Allocations",
    conn,
    if_exists="replace",
    index=False
)

timesheet.to_sql(
    "Timesheet",
    conn,
    if_exists="replace",
    index=False
)

print("Database Created")

# =====================================================
# CSV EXPORTS
# =====================================================

employees.to_csv(
    "Employees.csv",
    index=False
)

projects.to_csv(
    "Projects.csv",
    index=False
)

allocations.to_csv(
    "Allocations.csv",
    index=False
)

timesheet.to_csv(
    "Timesheet.csv",
    index=False
)

print("CSV Export Completed")

# =====================================================
# VALIDATION
# =====================================================

print("\n==============================")
print("PERSONA DISTRIBUTION")
print("==============================")

print(
    employees["Persona"].value_counts()
)

print("\n==============================")
print("UTILIZATION SUMMARY")
print("==============================")

print(
    employees["UtilizationPct"].describe()
)

print("\n==============================")
print("ATTRITION")
print("==============================")

print(
    employees["AttritionRisk"].value_counts()
)

print("\n==============================")
print("BURNOUT")
print("==============================")

print(
    employees["BurnoutRisk"].value_counts()
)

print("\n==============================")
print("PROJECT SUMMARY")
print("==============================")

print(
    "Employees :",
    len(employees)
)

print(
    "Projects :",
    len(projects)
)

print(
    "Allocations :",
    len(allocations)
)

print(
    "Timesheets :",
    len(timesheet)
)

print("\nPHASE 1 COMPLETED SUCCESSFULLY")

Employees Created
Projects Created
Allocations Created
Timesheet Created
Database Created
CSV Export Completed

PERSONA DISTRIBUTION
Persona
Average           2232
High Performer    1029
Bench              740
Future Leader      520
Burnout Risk       479
Name: count, dtype: int64

UTILIZATION SUMMARY
count    5000.000000
mean       70.899031
std        19.613736
min        23.072917
25%        63.333333
50%        67.395833
75%        90.989583
max        96.770833
Name: UtilizationPct, dtype: float64

ATTRITION
AttritionRisk
0    4260
1     740
Name: count, dtype: int64

BURNOUT
BurnoutRisk
0    3568
1    1432
Name: count, dtype: int64

PROJECT SUMMARY
Employees : 5000
Projects : 1000
Allocations : 9891
Timesheets : 60000

PHASE 1 COMPLETED SUCCESSFULLY


In [ ]:
# =====================================================
# PART 2 (FIXED VERSION)
# AI MODELS + PROBABILITIES + TALENT INTELLIGENCE
# =====================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

print("PART 2 STARTED")

# =====================================================
# FEATURE ENGINEERING
# =====================================================

employee_features = employees.copy()

# Project Count
project_count = allocations.groupby(
    "EmployeeID"
).size().reset_index(name="ProjectCount")

employee_features = employee_features.merge(
    project_count,
    on="EmployeeID",
    how="left"
)

employee_features["ProjectCount"] = employee_features[
    "ProjectCount"
].fillna(0)

# Average Allocation
avg_alloc = allocations.groupby(
    "EmployeeID"
)["AllocationPct"].mean().reset_index()

employee_features = employee_features.merge(
    avg_alloc,
    on="EmployeeID",
    how="left"
)

employee_features["AllocationPct"] = employee_features[
    "AllocationPct"
].fillna(0)

# =====================================================
# REALISTIC LABEL CREATION
# =====================================================

attr_score = (

    (employee_features["PerformanceRating"] < 3.2).astype(int)

    +

    (employee_features["TrainingHours"] < 15).astype(int)

    +

    (employee_features["Salary"] <
     employee_features["Salary"].median()).astype(int)

    +

    (employee_features["ExperienceYears"] < 3).astype(int)

)

employee_features["AttritionRisk"] = np.where(
    attr_score >= 3,
    1,
    0
)

burn_score = (

    (employee_features["UtilizationPct"] > 90).astype(int)

    +

    (employee_features["ProjectCount"] >= 3).astype(int)

    +

    (employee_features["TrainingHours"] < 15).astype(int)

)

employee_features["BurnoutRisk"] = np.where(
    burn_score >= 2,
    1,
    0
)

employee_features["BenchRisk"] = np.where(
    employee_features["UtilizationPct"] < 50,
    1,
    0
)

# =====================================================
# ENCODING
# =====================================================

encoders = {}

for col in [
    "Grade",
    "Skill",
    "Location",
    "Persona"
]:

    le = LabelEncoder()

    employee_features[col+"_Enc"] = le.fit_transform(
        employee_features[col]
    )

    encoders[col] = le

# =====================================================
# FEATURES
# =====================================================

feature_cols = [

    "ExperienceYears",
    "Salary",
    "PerformanceRating",
    "TrainingHours",
    "ProjectCount",
    "AllocationPct",

    "Grade_Enc",
    "Skill_Enc",
    "Location_Enc",
    "Persona_Enc"

]

# =====================================================
# GENERIC MODEL FUNCTION
# =====================================================

def build_model(df,target):

    X = df[feature_cols]

    y = df[target]

    X_train,X_test,y_train,y_test = train_test_split(

        X,
        y,

        test_size=0.20,

        random_state=42,

        stratify=y

    )

    model = RandomForestClassifier(

        n_estimators=200,

        max_depth=8,

        random_state=42

    )

    model.fit(

        X_train,
        y_train

    )

    pred = model.predict(X_test)

    acc = accuracy_score(

        y_test,
        pred

    )

    prob = model.predict_proba(X)[:,1]

    return model,acc,prob

# =====================================================
# ATTRITION MODEL
# =====================================================

attr_model,attr_acc,attr_prob = build_model(

    employee_features,

    "AttritionRisk"

)

employee_features[
    "AttritionProbability"
] = attr_prob

# =====================================================
# BURNOUT MODEL
# =====================================================

burn_model,burn_acc,burn_prob = build_model(

    employee_features,

    "BurnoutRisk"

)

employee_features[
    "BurnoutProbability"
] = burn_prob

# =====================================================
# BENCH MODEL
# =====================================================

bench_model,bench_acc,bench_prob = build_model(

    employee_features,

    "BenchRisk"

)

employee_features[
    "BenchRiskProbability"
] = bench_prob

# =====================================================
# FUTURE AVAILABILITY
# =====================================================

employee_features[
    "FutureAvailabilityPct"
] = (

    100

    -

    employee_features[
        "AllocationPct"
    ]

)

# =====================================================
# RESOURCE HEALTH
# =====================================================

employee_features[
    "ResourceHealthScore"
] = (

    employee_features[
        "PerformanceRating"
    ] * 20

    +

    employee_features[
        "UtilizationPct"
    ] * 0.4

    +

    employee_features[
        "TrainingHours"
    ] * 0.3

)

# =====================================================
# TALENT CATEGORY
# =====================================================

employee_features["TalentCategory"] = np.select(

    [

        (
            employee_features[
                "PerformanceRating"
            ] >= 4.5
        )

        &

        (
            employee_features[
                "UtilizationPct"
            ] >= 85
        ),

        (
            employee_features[
                "AttritionProbability"
            ] >= 0.70
        ),

        (
            employee_features[
                "BurnoutProbability"
            ] >= 0.70
        )

    ],

    [

        "Top Talent",

        "Attrition Threat",

        "Burnout Threat"

    ],

    default="Core Workforce"

)

# =====================================================
# BENCH CATEGORY
# =====================================================

employee_features["BenchCategory"] = pd.cut(

    employee_features[
        "UtilizationPct"
    ],

    bins=[0,50,75,100],

    labels=[
        "High Risk",
        "Medium Risk",
        "Healthy"
    ]

)

# =====================================================
# EXPORT
# =====================================================

employee_features.to_csv(

    "Employee_AI_Features.csv",

    index=False

)

# =====================================================
# RESULTS
# =====================================================

print("="*60)
print("MODEL ACCURACY")
print("="*60)

print(
    "Attrition Accuracy:",
    round(attr_acc,3)
)

print(
    "Burnout Accuracy:",
    round(burn_acc,3)
)

print(
    "Bench Accuracy:",
    round(bench_acc,3)
)

print("="*60)

print(
    employee_features[
        [
            "EmployeeID",
            "Persona",
            "AttritionProbability",
            "BurnoutProbability",
            "BenchRiskProbability",
            "TalentCategory"
        ]
    ].head()
)

print("="*60)
print("PART 2 COMPLETED")
print("="*60)

PART 2 STARTED
MODEL ACCURACY
Attrition Accuracy: 0.999
Burnout Accuracy: 0.99
Bench Accuracy: 1.0
  EmployeeID         Persona  AttritionProbability  BurnoutProbability  \
0     E00001         Average              0.000030            0.000000   
1     E00002    Burnout Risk              0.000147            0.994825   
2     E00003           Bench              0.067300            0.000000   
3     E00004         Average              0.000000            0.000000   
4     E00005  High Performer              0.000000            0.902835   

   BenchRiskProbability  TalentCategory  
0              0.000000  Core Workforce  
1              0.000000      Top Talent  
2              0.999911  Core Workforce  
3              0.000000  Core Workforce  
4              0.000000      Top Talent  
PART 2 COMPLETED


In [ ]:
# =====================================================
# PART 3
# FORECASTING + EXECUTIVE ANALYTICS
# =====================================================

!pip install -q plotly

import pandas as pd
import numpy as np
import plotly.express as px

print("PART 3 STARTED")

# =====================================================
# EXECUTIVE KPIs
# =====================================================

total_employees = len(employee_features)

total_projects = len(projects)

avg_utilization = round(

    employee_features[
        "UtilizationPct"
    ].mean(),

    2

)

avg_health = round(

    employee_features[
        "ResourceHealthScore"
    ].mean(),

    2

)

total_revenue = projects[
    "Revenue"
].sum()

total_cost = projects[
    "Cost"
].sum()

total_margin = (

    total_revenue

    -

    total_cost

)

Executive_KPI = pd.DataFrame({

    "Metric":[

        "Employees",
        "Projects",
        "Avg Utilization",
        "Resource Health",
        "Revenue",
        "Cost",
        "Margin"

    ],

    "Value":[

        total_employees,

        total_projects,

        avg_utilization,

        avg_health,

        total_revenue,

        total_cost,

        total_margin

    ]

})

print(Executive_KPI)

# =====================================================
# RESOURCE DEMAND FORECAST
# =====================================================

months = pd.date_range(

    "2024-01-01",

    periods=36,

    freq="M"

)

resource_demand = pd.DataFrame({

    "Month":months,

    "Demand":

    np.linspace(
        300,
        900,
        36
    )

    +

    np.random.randint(
        -50,
        50,
        36
    )

})

# Next 12 Months Forecast

future_months = pd.date_range(

    resource_demand["Month"].max(),

    periods=13,

    freq="M"

)[1:]

future_demand = pd.DataFrame({

    "Month":future_months,

    "Demand":

    np.linspace(

        resource_demand["Demand"].iloc[-1],

        resource_demand["Demand"].iloc[-1] + 250,

        12

    )

})

Demand_Forecast = pd.concat(

    [

        resource_demand,

        future_demand

    ]

)

print("Demand Forecast Created")

# =====================================================
# REVENUE FORECAST
# =====================================================

revenue_trend = pd.DataFrame({

    "Month":months,

    "Revenue":

    np.linspace(

        10000000,

        50000000,

        36

    )

    +

    np.random.randint(

        -5000000,

        5000000,

        36

    )

})

print("Revenue Forecast Created")

# =====================================================
# SKILL GAP ANALYSIS
# =====================================================

skill_current = employees.groupby(
    "Skill"
).size().reset_index(
    name="CurrentResources"
)

skill_current[
    "FutureDemand"
] = np.random.randint(

    200,

    1200,

    len(skill_current)

)

skill_current[
    "Gap"
] = (

    skill_current[
        "FutureDemand"
    ]

    -

    skill_current[
        "CurrentResources"
    ]

)

skill_demand = skill_current.copy()

print("Skill Gap Analysis Created")

# =====================================================
# SKILL RECOMMENDATION ENGINE
# =====================================================

skill_map = {

    "Excel":[
        "SQL, Power BI, Python"
    ],

    "SQL":[
        "Power BI, Python, Fabric"
    ],

    "Power BI":[
        "Fabric, Azure, Python"
    ],

    "Python":[
        "Databricks, Azure, ML"
    ],

    "Azure":[
        "Fabric, Databricks, Snowflake"
    ]

}

recommendations = []

for skill in employee_features[
    "Skill"
]:

    recommendations.append(

        skill_map.get(

            skill,

            ["Power BI, Python, Fabric"]

        )[0]

    )

employee_features[
    "RecommendedSkills"
] = recommendations

print("Recommendation Engine Ready")

# =====================================================
# TOP TALENT
# =====================================================

TopTalent = employee_features[

    employee_features[
        "TalentCategory"
    ] == "Top Talent"

]

# =====================================================
# ATTRITION THREAT
# =====================================================

AttritionThreat = employee_features[

    employee_features[
        "TalentCategory"
    ] == "Attrition Threat"

]

# =====================================================
# BURNOUT THREAT
# =====================================================

BurnoutThreat = employee_features[

    employee_features[
        "TalentCategory"
    ] == "Burnout Threat"

]

# =====================================================
# VISUALIZATION 1
# WORKFORCE SEGMENTATION
# =====================================================

fig = px.pie(

    employees,

    names="Persona",

    title="Workforce Segmentation"

)

fig.show()

# =====================================================
# VISUALIZATION 2
# UTILIZATION BY PERSONA
# =====================================================

fig = px.violin(

    employees,

    x="Persona",

    y="UtilizationPct",

    box=True,

    title="Utilization Distribution"

)

fig.show()

# =====================================================
# VISUALIZATION 3
# SALARY VS PERFORMANCE
# =====================================================

fig = px.scatter(

    employees,

    x="Salary",

    y="PerformanceRating",

    color="Persona",

    size="UtilizationPct",

    title="Salary vs Performance"

)

fig.show()

# =====================================================
# VISUALIZATION 4
# ATTRITION RISK
# =====================================================

fig = px.histogram(

    employee_features,

    x="AttritionProbability",

    nbins=30,

    title="Attrition Probability Distribution"

)

fig.show()

# =====================================================
# VISUALIZATION 5
# BURNOUT RISK
# =====================================================

fig = px.histogram(

    employee_features,

    x="BurnoutProbability",

    nbins=30,

    title="Burnout Probability Distribution"

)

fig.show()

# =====================================================
# VISUALIZATION 6
# REVENUE VS MARGIN
# =====================================================

fig = px.scatter(

    projects,

    x="Revenue",

    y="MarginPct",

    color="Priority",

    size="Revenue",

    title="Revenue vs Margin"

)

fig.show()

# =====================================================
# VISUALIZATION 7
# SKILL GAP
# =====================================================

fig = px.bar(

    skill_demand.sort_values(
        "Gap",
        ascending=False
    ),

    x="Skill",

    y="Gap",

    title="Future Skill Demand Gap"

)

fig.show()

# =====================================================
# VISUALIZATION 8
# DEMAND FORECAST
# =====================================================

fig = px.line(

    Demand_Forecast,

    x="Month",

    y="Demand",

    title="Resource Demand Forecast"

)

fig.show()

# =====================================================
# EXPORTS
# =====================================================

Executive_KPI.to_csv(
    "Executive_KPI.csv",
    index=False
)

Demand_Forecast.to_csv(
    "Demand_Forecast.csv",
    index=False
)

skill_demand.to_csv(
    "Skill_Demand.csv",
    index=False
)

employee_features.to_csv(
    "Employee_AI_Features.csv",
    index=False
)

print("="*60)
print("PART 3 COMPLETED")
print("="*60)

print("Files Created:")
print("Executive_KPI.csv")
print("Demand_Forecast.csv")
print("Skill_Demand.csv")
print("Employee_AI_Features.csv")

PART 3 STARTED
            Metric         Value
0        Employees  5.000000e+03
1         Projects  1.000000e+03
2  Avg Utilization  7.090000e+01
3  Resource Health  1.198800e+02
4          Revenue  7.640739e+10
5             Cost  5.351371e+10
6           Margin  2.289368e+10
Demand Forecast Created
Revenue Forecast Created
Skill Gap Analysis Created
Recommendation Engine Ready


/tmp/ipykernel_1820/870469163.py:100: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  months = pd.date_range(
/tmp/ipykernel_1820/870469163.py:134: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  future_months = pd.date_range(


PART 3 COMPLETED
Files Created:
Executive_KPI.csv
Demand_Forecast.csv
Skill_Demand.csv
Employee_AI_Features.csv


In [ ]:
# =====================================================
# PART 4
# EXECUTIVE COMMAND CENTER
# BIG 4 STYLE ANALYTICS
# =====================================================

import plotly.express as px
import plotly.graph_objects as go

print("PART 4 STARTED")

# =====================================================
# KPI SUMMARY
# =====================================================

print("="*60)
print("EXECUTIVE COMMAND CENTER")
print("="*60)

print("Employees :", len(employee_features))
print("Projects :", len(projects))

print(
    "Avg Utilization :",
    round(
        employee_features["UtilizationPct"].mean(),
        2
    )
)

print(
    "Revenue : ₹",
    round(
        projects["Revenue"].sum()/10000000,
        2
    ),
    "Cr"
)

print(
    "Margin : ₹",
    round(
        (
            projects["Revenue"].sum()
            -
            projects["Cost"].sum()
        )/10000000,
        2
    ),
    "Cr"
)

# =====================================================
# WORKFORCE PYRAMID
# =====================================================

grade_count = employees.groupby(
    "Grade"
).size().reset_index(
    name="Count"
)

fig = px.funnel(
    grade_count,
    x="Count",
    y="Grade",
    title="Workforce Pyramid"
)

fig.show()

# =====================================================
# TALENT MATRIX
# =====================================================

fig = px.scatter(

    employee_features,

    x="PerformanceRating",

    y="UtilizationPct",

    color="TalentCategory",

    size="Salary",

    hover_data=[
        "EmployeeID",
        "Skill",
        "Location"
    ],

    title="Talent Matrix"
)

fig.show()

# =====================================================
# UTILIZATION HEATMAP
# =====================================================

heat = employees.pivot_table(

    values="UtilizationPct",

    index="Location",

    columns="Grade",

    aggfunc="mean"

)

fig = px.imshow(

    heat,

    title="Utilization Heatmap"

)

fig.show()

# =====================================================
# BENCH INTELLIGENCE
# =====================================================

bench_summary = employee_features.groupby(

    "BenchCategory"

).size().reset_index(
    name="Employees"
)

fig = px.pie(

    bench_summary,

    names="BenchCategory",

    values="Employees",

    title="Bench Intelligence"

)

fig.show()

# =====================================================
# ATTRITION INTELLIGENCE
# =====================================================

attr_loc = employee_features.groupby(

    "Location"

)["AttritionProbability"].mean().reset_index()

fig = px.bar(

    attr_loc,

    x="Location",

    y="AttritionProbability",

    title="Attrition Risk by Location"

)

fig.show()

# =====================================================
# BURNOUT INTELLIGENCE
# =====================================================

burn_skill = employee_features.groupby(

    "Skill"

)["BurnoutProbability"].mean().reset_index()

fig = px.bar(

    burn_skill.sort_values(
        "BurnoutProbability",
        ascending=False
    ),

    x="Skill",

    y="BurnoutProbability",

    title="Burnout Risk by Skill"

)

fig.show()

# =====================================================
# REVENUE ANALYTICS
# =====================================================

fig = px.scatter(

    projects,

    x="Revenue",

    y="MarginPct",

    size="Revenue",

    color="Priority",

    title="Revenue vs Margin"

)

fig.show()

# =====================================================
# SKILL GAP ANALYTICS
# =====================================================

fig = px.bar(

    skill_demand.sort_values(
        "Gap",
        ascending=False
    ),

    x="Skill",

    y="Gap",

    title="Skill Gap Analysis"

)

fig.show()

# =====================================================
# FUTURE DEMAND
# =====================================================

fig = px.line(

    Demand_Forecast,

    x="Month",

    y="Demand",

    title="Future Resource Demand"

)

fig.show()

# =====================================================
# TOP TALENT
# =====================================================

top_talent = employee_features[

    employee_features[
        "TalentCategory"
    ] == "Top Talent"

]

print("\nTOP TALENT")

display(

top_talent[

    [
        "EmployeeID",
        "Skill",
        "Location",
        "PerformanceRating",
        "UtilizationPct"
    ]

].head(20)

)

# =====================================================
# ATTRITION THREAT
# =====================================================

attrition_threat = employee_features.sort_values(

    "AttritionProbability",

    ascending=False

).head(20)

print("\nATTRITION THREAT")

display(

attrition_threat[

    [
        "EmployeeID",
        "Skill",
        "Location",
        "AttritionProbability"
    ]

]

)

# =====================================================
# BURNOUT THREAT
# =====================================================

burnout_threat = employee_features.sort_values(

    "BurnoutProbability",

    ascending=False

).head(20)

print("\nBURNOUT THREAT")

display(

burnout_threat[

    [
        "EmployeeID",
        "Skill",
        "Location",
        "BurnoutProbability"
    ]

]

)

# =====================================================
# EXECUTIVE AI INSIGHTS
# =====================================================

print("\n")
print("="*60)
print("AI INSIGHTS")
print("="*60)

print(
    "Highest Demand Skill :",
    skill_demand.sort_values(
        "Gap",
        ascending=False
    ).iloc[0]["Skill"]
)

print(
    "Highest Burnout Skill :",
    burn_skill.sort_values(
        "BurnoutProbability",
        ascending=False
    ).iloc[0]["Skill"]
)

print(
    "Highest Attrition Location :",
    attr_loc.sort_values(
        "AttritionProbability",
        ascending=False
    ).iloc[0]["Location"]
)

print(
    "Top Talent Count :",
    len(top_talent)
)

print(
    "Attrition Threat Count :",
    len(
        employee_features[
            employee_features[
                "AttritionProbability"
            ] > 0.7
        ]
    )
)

print(
    "Burnout Threat Count :",
    len(
        employee_features[
            employee_features[
                "BurnoutProbability"
            ] > 0.7
        ]
    )
)

print("="*60)

# =====================================================
# FINAL EXPORTS
# =====================================================

top_talent.to_csv(
    "Top_Talent.csv",
    index=False
)

attrition_threat.to_csv(
    "Attrition_Threat.csv",
    index=False
)

burnout_threat.to_csv(
    "Burnout_Threat.csv",
    index=False
)

print("PART 4 COMPLETED")

PART 4 STARTED
EXECUTIVE COMMAND CENTER
Employees : 5000
Projects : 1000
Avg Utilization : 70.9
Revenue : ₹ 7640.74 Cr
Margin : ₹ 2289.37 Cr


/tmp/ipykernel_1820/2138970456.py:129: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.




TOP TALENT


,EmployeeID,Skill,Location,PerformanceRating,UtilizationPct
1,E00002,Python,Bangalore,4.7,94.947917
4,E00005,Data Analytics,Kolkata,4.6,90.104167
6,E00007,Tableau,Mumbai,4.7,89.947917
11,E00012,Alteryx,Kolkata,4.8,93.697917
14,E00015,Databricks,Pune,4.6,91.614583
15,E00016,Alteryx,Bangalore,4.7,91.927083
29,E00030,Tableau,Mumbai,4.5,89.791667
31,E00032,Power BI,Gurugram,4.8,91.562500
32,E00033,Azure,Chennai,4.5,90.885417
33,E00034,Business Analysis,Mumbai,5.0,93.958333



ATTRITION THREAT


,EmployeeID,Skill,Location,AttritionProbability
2321,E02322,Excel,Chennai,0.999615
157,E00158,Databricks,Gurugram,0.999210
1625,E01626,Power BI,Bangalore,0.997558
419,E00420,Data Analytics,Hyderabad,0.997475
1272,E01273,Fabric,Bangalore,0.997169
4762,E04763,SQL,Hyderabad,0.996650
4037,E04038,SQL,Bangalore,0.996640
3373,E03374,Excel,Bangalore,0.996615
199,E00200,SQL,Kolkata,0.996463
3310,E03311,Databricks,Pune,0.995958



BURNOUT THREAT


,EmployeeID,Skill,Location,BurnoutProbability
2646,E02647,Databricks,Bangalore,1.000000
2736,E02737,Snowflake,Pune,1.000000
4144,E04145,Excel,Chennai,1.000000
4256,E04257,Excel,Mumbai,1.000000
1727,E01728,SQL,Hyderabad,1.000000
3636,E03637,Snowflake,Chennai,0.999935
3132,E03133,Python,Chennai,0.999918
1698,E01699,Alteryx,Chennai,0.999918
3920,E03921,Alteryx,Pune,0.999894
4840,E04841,Alteryx,Pune,0.999894




AI INSIGHTS
Highest Demand Skill : Excel
Highest Burnout Skill : Power BI
Highest Attrition Location : Chennai
Top Talent Count : 1001
Attrition Threat Count : 405
Burnout Threat Count : 986
PART 4 COMPLETED
